# 05 — CNN (Stage D, Arm 2)

EfficientNet-B0 (ImageNet-pretrained) with a shared encoder and two heads: six flaws (BLR BRT DRK
FRM OBS ROT at >=2 votes) and UNREC (unrecognizable >=2). Trained by `scripts/train_cnn.py` on the
224px letterboxed image cache, in three runs: an overfit check on 200 images, a frozen-backbone run
and a full fine-tune. This notebook trains nothing: it reads the saved val scores and training
histories, compares the CNN with the classical arm, and regenerates `reports/results_cnn.md` and
`reports/results_cnn.csv`.

**Partitions:** trained on train; best epoch and early stopping chosen on val; scored on val. The
test partition is never read.

**How to run:** from the `notebooks/` folder, with `DATA_DIR` pointing at the data directory
(defaults to `../data`). `DATA_DIR` must contain `annotations/{train,val}.json`,
`image_properties.csv`, `predictions/classical_gb_224_val.csv` (Stage C),
`predictions/cnn_{frozen,finetune}_val.csv` and `cnn_runs/{overfit,frozen,finetune}/history.json`
(from `scripts/train_cnn.py`). No GPU needed; runs in seconds.

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
from IPython.display import display

REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
sys.path.insert(0, str(REPO_DIR))
DATA_DIR = Path(os.environ.get("DATA_DIR", REPO_DIR / "data"))
REPORT_PATH = REPO_DIR / "reports" / "results_cnn.md"
RESULTS_CSV = REPO_DIR / "reports" / "results_cnn.csv"
FIGURES_DIR = REPO_DIR / "reports" / "figures"
PREDICTIONS_DIR = DATA_DIR / "predictions"
RUNS_DIR = DATA_DIR / "cnn_runs"

from src.data.splits import MODELLED_FLAWS, get_splits
from src.eda import image_props as ip
from src.eda import labels as lb
from src.eda import report as rp
from src.models import classical as cm
from src.models import metrics as mt

CLASSICAL = "Classical GB @ 224"
FROZEN = "CNN frozen"
FINETUNE = "CNN fine-tuned"   # the CNN arm's entry in the Stage F comparison
PREDICTION_FILES = {CLASSICAL: "classical_gb_224_val.csv", FROZEN: "cnn_frozen_val.csv",
                    FINETUNE: "cnn_finetune_val.csv"}
FLAW_TYPE = {"BLR": "photometric", "DRK": "photometric", "BRT": "photometric",
             "FRM": "semantic", "OBS": "semantic", "ROT": "semantic", cm.UNREC: "recognisability"}
ENVIRONMENT = rp.environment_lines(
    gpu_note="not needed here: this notebook scores saved predictions; the training GPU is under "
             "Training runs",
    seed_note="training seeded torch/numpy/random with 42, reseeded 42 + epoch every epoch")
pd.set_option("display.precision", 3)

print("REPO_DIR:", REPO_DIR)
print("DATA_DIR:", DATA_DIR)
print("\n".join(ENVIRONMENT))

## 1. Data
Targets come from the annotation records via `src/data/splits.py`. Each prediction file must hold
exactly the val images, with one score column per output.

In [ ]:
splits = get_splits(DATA_DIR)
targets = cm.build_targets(splits["val"])
scores = {}
for name, filename in PREDICTION_FILES.items():
    s = pd.read_csv(PREDICTIONS_DIR / filename, index_col="image")
    assert sorted(s.index) == sorted(targets.index), f"{filename}: image set differs from val"
    assert list(s.columns) == cm.OUTPUTS, f"{filename}: columns {list(s.columns)}"
    scores[name] = s.loc[targets.index]
    print(f"{name:<20} {filename:<26} {len(s):,} images, scores {s.values.min():.3f}-{s.values.max():.3f}")
sections = []  # (heading, markdown) pairs collected for the report

positives = pd.DataFrame({"val positives": targets.sum(), "base rate": targets.mean()})
display(positives)
sections.append(("Data", (
    f"Val {len(targets):,} images, scored by every arm. Positives per output at >=2 votes:\n\n"
    + rp.df_to_markdown(positives.round(3)))))

## 2. Training runs
Read from each run's `history.json`, written by `scripts/train_cnn.py` on Colab.

In [ ]:
history = {mode: json.loads((RUNS_DIR / mode / "history.json").read_text())
           for mode in ("overfit", "frozen", "finetune")}
runs, environments = {}, {}
for mode, h in history.items():
    c = h["config"]
    epochs = pd.DataFrame(h["epochs"])
    if mode == "overfit":
        result = (f"{h['overfit']['verdict']}: fit loss {h['overfit']['final_fit_loss']:.4f} "
                  f"(target < {h['overfit']['target']})")
    else:
        result = f"best val macro AP {h['best_val_macro_ap']:.4f}"
    runs[mode] = {
        "data": "200 fixed train images, no flip" if mode == "overfit" else "full train, horizontal flip",
        "trains": "heads only" if mode == "frozen" else "whole network",
        "lr heads / backbone": f"{c['lr_head']:g} / " + (f"{c['lr_backbone']:g}" if c["lr_backbone"] else "frozen"),
        "max epochs": c["max_epochs"],
        "epochs run": len(epochs),
        "best epoch": h["best_epoch"] if h["best_epoch"] is not None else "-",
        "s per epoch": round(epochs["seconds"].median()),
        "result": result,
    }
    environments[mode] = h["environment"]
runs = pd.DataFrame(runs).T
environments = pd.DataFrame(environments).T
display(runs)
display(environments)
c = history["finetune"]["config"]
sections.append(("Training runs", (
    f"`scripts/train_cnn.py`: {c['backbone']}, batch {c['batch_size']}, AdamW (default weight decay), "
    "BCE with pos_weight = negatives/positives on train, mixed precision, constant learning rates, "
    f"early stop after {c['patience']} epochs without improvement in val macro AP (six flaws), "
    f"seed {c['seed']}.\n\n" + rp.df_to_markdown(runs) + "\n\nTraining environment:\n\n"
    + rp.df_to_markdown(environments))))

## 3. Training curves
Train loss and val macro AP per epoch. Where train loss keeps falling while val AP stops
improving, the model has started fitting the training images rather than general cues; early
stopping keeps the best val epoch.

In [ ]:
fig, (ax_loss, ax_ap) = plt.subplots(1, 2, figsize=(11, 4))
for mode, label in (("frozen", FROZEN), ("finetune", FINETUNE)):
    e = pd.DataFrame(history[mode]["epochs"])
    ax_loss.plot(e["epoch"], e["train_loss"], marker="o", label=label)
    ax_ap.plot(e["epoch"], e["val_macro_ap"], marker="o", label=label)
    best = history[mode]["best_epoch"]
    ax_ap.scatter([best], e.set_index("epoch").loc[[best], "val_macro_ap"], s=160,
                  facecolors="none", edgecolors="black", zorder=3)
classical_macro = mt.evaluate(targets, scores[CLASSICAL]).loc[mt.MACRO, "AP"]
ax_ap.axhline(classical_macro, color="grey", linestyle="--", label=f"{CLASSICAL} ({classical_macro:.3f})")
ax_loss.set(title="Train loss (weighted BCE)", xlabel="epoch", ylabel="loss")
ax_ap.set(title="Val macro AP, six flaws (circle = best epoch)", xlabel="epoch", ylabel="AP")
for ax in (ax_loss, ax_ap):
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend()
    ax.grid(alpha=0.3)
fig.tight_layout()
figure = rp.save(fig, FIGURES_DIR, "cnn_training_curves")
plt.show()

per_epoch = pd.DataFrame([{"epoch": r["epoch"], "train loss": r["train_loss"],
                           "val macro AP": r["val_macro_ap"], **r["val_ap"]}
                          for r in history["finetune"]["epochs"]]).set_index("epoch")
display(per_epoch)
sections.append(("Training curves", (
    f"![Training curves](figures/{figure.name})\n\nFine-tune, per epoch (val AP):\n\n"
    + rp.df_to_markdown(per_epoch.round(3)))))

## 4. CNN results on val
AP is the primary metric and is threshold-free; **read it against the base rate** (the AP of a
random scorer). Gain = (AP - base rate) / (1 - base rate) puts outputs with different base rates
on one scale. Precision, recall and F1 are at a fixed 0.5 threshold. The macro row averages the six
flaws; UNREC is the separate recognisability head.

In [ ]:
results = {name: mt.evaluate(targets, s) for name, s in scores.items()}
for table in results.values():
    table.insert(table.columns.get_loc("AP") + 1, "gain", mt.normalised_gain(table["AP"], table["base rate"]))
    table.loc[mt.MACRO, "gain"] = np.nan  # gain is read per output, not on the averaged row
for name in (FINETUNE, FROZEN):
    print(name)
    display(results[name])

long = pd.concat({name: results[name] for name in (FROZEN, FINETUNE)}, names=["config", "output"])
RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
long.reset_index().to_csv(RESULTS_CSV, index=False, float_format="%.4f")
for name in (FINETUNE, FROZEN):
    sections.append((f"Full table: {name}", rp.df_to_markdown(results[name].round(3))))

## 5. CNN against the classical arm (decision point)
Classical entry GB @ 224 (Stage C) against the CNN entry (fine-tuned), same val images, same
metrics code. The hypothesis predicts the CNN's gain is concentrated in the semantic flaws
(FRM/OBS/ROT).

In [ ]:
rows = cm.OUTPUTS + [mt.MACRO]
compare = pd.DataFrame({
    "type": [FLAW_TYPE.get(o, "") for o in rows],
    "base rate": results[FINETUNE].loc[rows, "base rate"],
    f"AP {CLASSICAL}": results[CLASSICAL].loc[rows, "AP"],
    f"AP {FROZEN}": results[FROZEN].loc[rows, "AP"],
    f"AP {FINETUNE}": results[FINETUNE].loc[rows, "AP"],
    f"gain {CLASSICAL}": results[CLASSICAL].loc[rows, "gain"],
    f"gain {FINETUNE}": results[FINETUNE].loc[rows, "gain"],
}, index=rows)
compare["gain difference (CNN - classical)"] = compare[f"gain {FINETUNE}"] - compare[f"gain {CLASSICAL}"]
display(compare)

by_type = compare.loc[MODELLED_FLAWS].groupby("type")[
    [f"gain {CLASSICAL}", f"gain {FINETUNE}", "gain difference (CNN - classical)"]].mean()
print("Mean gain by flaw type")
display(by_type)

f1 = pd.DataFrame({name: results[name].loc[rows, "F1"] for name in PREDICTION_FILES})
print("F1 @ 0.5")
display(f1)
sections.append(("CNN against the classical arm (val)", (
    "AP and gain per output; gain = (AP - base rate) / (1 - base rate).\n\n"
    + rp.df_to_markdown(compare.round(3)) + "\n\nMean gain by flaw type:\n\n"
    + rp.df_to_markdown(by_type.round(3)) + "\n\nF1 at threshold 0.5:\n\n"
    + rp.df_to_markdown(f1.round(3)))))

## 6. ROT: content or shape?
Landscape-shaped raw pixels predict ROT without any image content, and the letterboxed cache keeps
that shape visible to the CNN. Two bars from Stage C: ROT AP above the shape-only reference on all
val images, and above chance **within EXIF orientation 1**, where every image has the ordinary
shape and the cue carries no signal.

In [ ]:
props = ip.load_properties(DATA_DIR / "image_properties.csv")
val_props = props[props["image"].isin(targets.index)]
shape_ref = ip.rot_shortcut_scores(val_props, lb.vote_matrix(splits["val"]))
display(shape_ref)

orientation = props.set_index("image").loc[targets.index, "orientation"]
rot_rows = {}
for name, s in scores.items():
    row = {f"AP, all (n={len(targets)}, base {targets['ROT'].mean():.3f})":
           mt.average_precision(targets["ROT"], s["ROT"])}
    for tag in (1, 6):
        idx = orientation.index[orientation == tag]
        y = targets.loc[idx, "ROT"]
        ap = mt.average_precision(y, s.loc[idx, "ROT"])
        row[f"AP, orientation {tag} (n={len(idx)}, base {y.mean():.3f})"] = ap
        row[f"gain, orientation {tag}"] = mt.normalised_gain(ap, y.mean())
    rot_rows[name] = row
rot_by_orientation = pd.DataFrame(rot_rows).T
display(rot_by_orientation)
sections.append(("ROT: content or shape? (val)", (
    "Geometry-only cues scored as ROT predictors on val:\n\n" + rp.df_to_markdown(shape_ref)
    + "\n\nROT AP by EXIF orientation:\n\n" + rp.df_to_markdown(rot_by_orientation.round(3)))))

## 7. BLR on natively small images
Images whose original long edge is below 512 px carry less fine detail; the few below 224 px were
upscaled into the cache. BLR AP is split by native size for both arms, as in Stage C.

In [ ]:
native_long = props.set_index("image").loc[targets.index, ["width", "height"]].max(axis=1)
blr_rows = {}
for name, s in scores.items():
    row = {}
    for label, mask in (("native < 512", native_long < 512), ("native >= 512", native_long >= 512)):
        y = targets.loc[mask, "BLR"]
        row[f"{label} (n={int(mask.sum())}, base {y.mean():.3f})"] = mt.average_precision(y, s.loc[y.index, "BLR"])
    blr_rows[name] = row
blr_by_size = pd.DataFrame(blr_rows).T
display(blr_by_size)
sections.append(("BLR by native resolution (val)", rp.df_to_markdown(blr_by_size.round(3))))

## 8. Single-flaw check
AP for each flaw on images carrying no other modelled flaw (positives: that flaw alone; negatives:
no flaw). Flaws co-occur with blur (P(BLR | OBS) = 64% on train), so a model could score DRK, BRT
or OBS partly by detecting blur. If the CNN's advantage holds here, it responds to the flaw itself.
Small positive counts make some rows indicative only.

In [ ]:
single = {name: mt.single_flaw_ap(targets, s) for name, s in scores.items()}
first = next(iter(single.values()))
single_table = first[["n", "positives (flaw alone)", "base rate"]].copy()
for name, t in single.items():
    single_table[f"AP {name}"] = t["AP"]
for name in (CLASSICAL, FINETUNE):
    single_table[f"gain {name}"] = mt.normalised_gain(single_table[f"AP {name}"], single_table["base rate"])
display(single_table)
sections.append(("Single-flaw check (val)", rp.df_to_markdown(single_table.round(3))))

## 9. Write the report
The findings block in `reports/results_cnn.md` is hand-written after reading the results and is
preserved on every rerun.

In [ ]:
out = rp.write_report(REPORT_PATH, sections, title="Stage D: CNN (Arm 2)",
                      environment=ENVIRONMENT, findings_heading="Findings (hypothesis check)")
print("Report written to", out)
print("Results table written to", RESULTS_CSV)
print("Figure written to", figure)